# Exercise 3.2 - Parameter-efficient adaptation of CLIP

Kernel consigliato: `clip_lora`.

Obiettivo: valutare CLIP zero-shot su ImageNet-Sketch e poi adattarlo con un metodo parameter-efficient. Usiamo CLIP-Adapter sulle feature immagine perche' e' leggero, chiaro e adatto a GPU con poca VRAM.

Nota: questo notebook usa un adapter, non una LoRA interna al transformer. E' comunque parameter-efficient: il backbone CLIP resta congelato e si addestra solo un piccolo MLP.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT

In [ ]:
from torch.utils.data import DataLoader
import pandas as pd
import torch

from dla_lab2.clip_utils import (
    CLIPAdapter,
    build_clip_tensor_dataset,
    build_text_features,
    build_text_features_ensemble,
    evaluate_adapter,
    evaluate_zero_shot,
    load_imagenet_labels,
    load_imagenet_sketch,
    load_open_clip_model,
    precompute_image_features,
    split_tensor_dataset,
    train_clip_adapter,
)
from dla_lab2.seed import set_seed

set_seed(42)

## 1. Modello e labels

Carichiamo CLIP ViT-B/16 e i nomi semplici delle 1000 classi ImageNet. Usiamo la variante quickgelu per coerenza con i pesi OpenAI.

In [ ]:
model, preprocess, tokenizer, device = load_open_clip_model(
    model_name="ViT-B-16-quickgelu",
    pretrained="openai",
)

imagenet_class_names = load_imagenet_labels(PROJECT_ROOT / "imagenet_labels.json")
print(device)
print(len(imagenet_class_names), imagenet_class_names[:5])

## 2. Dataset ImageNet-Sketch

ImageNet-Sketch e' piu' interessante di ImageNette perche' introduce domain shift: CLIP vede disegni e schizzi invece di foto naturali.

In [ ]:
sketch_train, sketch_val = load_imagenet_sketch(seed=42, train_fraction=0.8)
print(len(sketch_train), len(sketch_val))

In [ ]:
TRAIN_SAMPLES = 5000
BATCH_SIZE = 64

train_tensor_ds = build_clip_tensor_dataset(sketch_train, preprocess, num_samples=TRAIN_SAMPLES)
val_tensor_ds = build_clip_tensor_dataset(sketch_val, preprocess, num_samples=None)

train_image_loader = DataLoader(train_tensor_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
val_image_loader = DataLoader(val_tensor_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

## 3. Zero-shot baseline e prompt study

Prima valutiamo CLIP senza training. Poi confrontiamo alcuni prompt semplici. Questo e' importante perche' CLIP e' molto sensibile al testo usato come descrizione della classe.

In [ ]:
prompt_templates = [
    "a sketch of a {}",
    "a black and white sketch of a {}",
    "a hand-drawn sketch of a {}",
    "a pencil drawing of a {}",
    "a line drawing of a {}",
]

prompt_results = []
for template in prompt_templates:
    text_features = build_text_features(model, tokenizer, imagenet_class_names, template, device)
    acc = evaluate_zero_shot(model, val_image_loader, text_features, device)
    prompt_results.append({"prompt": template, "accuracy": acc})

prompt_table = pd.DataFrame(prompt_results).sort_values("accuracy", ascending=False)
prompt_table

## 4. CLIP-Adapter

Congeliamo CLIP, precalcoliamo le feature immagine e addestriamo solo un adapter MLP. Questo riduce tempo, memoria e rischio di danneggiare le rappresentazioni zero-shot.

In [ ]:
train_feature_ds = precompute_image_features(model, train_image_loader, device)
adapter_train_ds, adapter_val_ds = split_tensor_dataset(train_feature_ds, train_fraction=0.9, seed=42)

adapter_train_loader = DataLoader(adapter_train_ds, batch_size=256, shuffle=True)
adapter_val_loader = DataLoader(adapter_val_ds, batch_size=256, shuffle=False)

val_feature_ds = precompute_image_features(model, val_image_loader, device)
val_feature_loader = DataLoader(val_feature_ds, batch_size=256, shuffle=False)

In [ ]:
base_prompt = prompt_table.iloc[0]["prompt"]
text_features_base = build_text_features(model, tokenizer, imagenet_class_names, base_prompt, device)
logit_scale = model.logit_scale.exp().item()

adapter64 = CLIPAdapter(feat_dim=512, bottleneck=64, alpha=0.6)
history64 = train_clip_adapter(
    adapter64,
    adapter_train_loader,
    adapter_val_loader,
    text_features_base,
    logit_scale,
    device,
    epochs=30,
    lr=2e-3,
)

adapter128 = CLIPAdapter(feat_dim=512, bottleneck=128, alpha=0.6)
history128 = train_clip_adapter(
    adapter128,
    adapter_train_loader,
    adapter_val_loader,
    text_features_base,
    logit_scale,
    device,
    epochs=30,
    lr=2e-3,
)

## 5. Valutazione finale

Valutiamo zero-shot, adapter singolo e adapter con prompt ensembling sul validation set esterno ImageNet-Sketch.

In [ ]:
zeroshot_acc = evaluate_zero_shot(model, val_image_loader, text_features_base, device)
acc64 = evaluate_adapter(adapter64, val_feature_loader, text_features_base, logit_scale, device)
acc128 = evaluate_adapter(adapter128, val_feature_loader, text_features_base, logit_scale, device)

text_features_ensemble = build_text_features_ensemble(model, tokenizer, imagenet_class_names, prompt_templates, device)
zs_ensemble = evaluate_zero_shot(model, val_image_loader, text_features_ensemble, device)
acc64_ensemble = evaluate_adapter(adapter64, val_feature_loader, text_features_ensemble, logit_scale, device)
acc128_ensemble = evaluate_adapter(adapter128, val_feature_loader, text_features_ensemble, logit_scale, device)

results = pd.DataFrame(
    [
        {"method": "Zero-shot CLIP", "accuracy": zeroshot_acc, "gain": 0.0},
        {"method": "Zero-shot CLIP + prompt ensemble", "accuracy": zs_ensemble, "gain": zs_ensemble - zeroshot_acc},
        {"method": "CLIP-Adapter bottleneck=64", "accuracy": acc64, "gain": acc64 - zeroshot_acc},
        {"method": "CLIP-Adapter b=64 + prompt ensemble", "accuracy": acc64_ensemble, "gain": acc64_ensemble - zeroshot_acc},
        {"method": "CLIP-Adapter bottleneck=128", "accuracy": acc128, "gain": acc128 - zeroshot_acc},
        {"method": "CLIP-Adapter b=128 + prompt ensemble", "accuracy": acc128_ensemble, "gain": acc128_ensemble - zeroshot_acc},
    ]
)
results

In [ ]:
pd.DataFrame(history64).assign(adapter="bottleneck=64").tail(), pd.DataFrame(history128).assign(adapter="bottleneck=128").tail()

## Conclusioni Exercise 3.2

- ImageNet-Sketch e' adatto per testare domain shift: CLIP parte gia' forte ma non e' perfetto.
- Il CLIP-Adapter addestra pochi parametri e lascia congelato il backbone. Se migliora, significa che le feature CLIP erano buone ma serviva un piccolo adattamento al dominio sketch.
- Se l'adapter overfitta, si vede da validation loss crescente: in quel caso si riducono epoche, bottleneck o learning rate.
- Nel report finale va specificato che la tecnica scelta e' parameter-efficient adapter, non LoRA interna al transformer.